# Proyecto CRUZBER — Conclusiones Hasta Iteración 8
## Predicción de Demanda Semanal por SKU y Localidad

### Documento Ejecutivo • Iteraciones 1-8 • Enero-Marzo 2026

---

> **El desafío:** CRUZBER vende cientos de productos en miles de localidades. Cada semana, el equipo de compras decide cuántas unidades de cada referencia enviar a cada zona sin datos precisos. Esta incertidumbre cuesta dinero: stock muerto que se queda en almacén o roturas de venta por falta de producto.
>
> **La solución que hemos construido:** Un sistema de predicción que usa el historial de ventas, el precio, el clima, los descuentos y la geografía para predecir con un **error del 18-21%** la demanda de la semana siguiente.
>
> **El impacto:** Pasar de "adivinar" con un 35-50% de error a predecir con un 18% de error significa comprar mejor, vender más y tener menos capital inmovilizado en stock.

---

## Resumen Ejecutivo en 60 segundos

### ¿Qué hemos hecho?

Hemos entrenado **dos sistemas de predicción de demanda** que trabajan juntos:

**Sistema 1: Predicción por localidad (It7)**
- Responde: ¿Cuánto vende SKU X en cada municipio la próxima semana?
- Error medio: **21.5%** (si predice 10 unidades, la realidad estará entre 8 y 12)
- Uso: decidir qué cantidad enviar a cada almacén local

**Sistema 2: Predicción por región (It8)**
- Responde: ¿Cuánto vende SKU X en toda la región Noroeste la próxima semana?
- Error medio: **18.3%** (más preciso porque la demanda agregada es más suave)
- Uso: decidir el volumen total de compra para cada zona, y validar que las predicciones locales son coherentes

### ¿Cuánto mejoramos respecto al punto de partida?

| Métrica | Inicio (It1) | Hoy (It7-It8) | Mejora |
|---|---|---|---|
| **Error promedio** | 0.793 unidades | 0.628 unidades | **−21%** |
| **Error relativo** | ~26% | ~18-21% | **−5 a 8 pp** |
| **Capacidad explicativa (R²)** | 0.295 | 0.264 (municipio) / 0.454 (región) | Variable |

### ¿Vale la pena?

**Sí.** El coste de equivocarse en la compra es mucho mayor que el coste de mantener dos modelos. Si cada error de 100 unidades cuesta €500 en sobrestock u oportunidad perdida, **reducir el error un 21% evita pérdidas de decenas de miles de euros al año**.

---

## El Viaje: 8 Iteraciones hacia la Precisión

### Iteración 1: El Primer Paso (Baseline)
**Qué hicimos:** Un modelo simple que aprende de los históricos de venta sin ningún ajuste especial.
**Resultado:** MAE 0.793. El modelo existe pero es muy básico.
**Lección:** Antes de mejorar, hay que saber desde dónde se parte.

### Iteraciones 2-3: Añadiendo Memoria
**Qué hicimos:** Le dijimos al modelo: "recuerda lo que pasó las últimas 4 semanas" y "recuerda qué pasó hace un año".
**Resultado:** MAE 0.773 → 0.769. Mejora muy pequeña.
**Lección:** A veces añadir más datos no ayuda si el modelo no sabe cómo usarlos.

### Iteración 4: El Gran Salto
**Qué hicimos:** Transformamos cómo el modelo ve los números. En vez de predecir "vendo 100 unidades", predice "vendo log(100)". Esto equilibra los productos pequeños y grandes en la misma escala.
**Resultado:** MAE **0.649**. Mejora del **−15.6%** en una sola iteración.
**Lección:** A veces la solución no es tener más datos, sino formular el problema diferente.

### Iteración 5: Historia Profunda
**Qué hicimos:** Creamos una variable que resume: "¿cuánto se ha vendido históricamente de este producto en esta localidad?" Esta variable se convierte en la **más importante del modelo (19% de influencia)**.
**Resultado:** MAE 0.641 (mejora del −1.2%). Pequeño avance, pero la nueva variable es la brújula del modelo.
**Lección:** El mejor predictor es el historial propio. Si sabes que algo siempre vende 15 unidades en ese lugar, ese número es oro puro.

### Iteración 6: Nuevas Fuentes de Señal
**Qué hicimos:** Incorporamos dos nuevas variables:
1. **Descuentos promocionales**: descubrimos que multiplican la demanda ×4.2
2. **Geografía regional**: agrupamos provincias en 6 regiones para ver si hay patrones por clima

**Resultado:** MAE sigue en 0.641 (sin mejora en MAE, pero R² mejora). Los descuentos importan, la región no tanto.
**Lección:** No todos los datos nuevos son útiles. El descuento sí (×4.2 es una señal potente), la región geográfica sola no (ya está implícita en "municipio").

### Iteración 7: Arquitectura Inteligente
**Qué hicimos:** En lugar de un modelo que intente ser bueno en todo, entrenamos **dos modelos especializados**:
- Modelo A: solo para productos de alta rotación (comportamiento estable)
- Modelo B&C: solo para productos de nicho (comportamiento esporádico)

**Resultado:** MAE **0.628**, MAPE **21.5%**. Mejora del −2% en MAE, pero MAPE baja un **−16.6%** (la mejor métrica de negocio).
**Lección:** "Un modelo para todos" es un compromiso que satisface a nadie. Dos modelos especializados funcionan mejor.

### Iteración 8: Vista de Helicóptero
**Qué hicimos:** Agregamos los datos de municipio a nivel región. Entrenamos modelos regionales. Pregunta: ¿prediciendo directamente a escala regional obtenemos más precisión?

**Resultado:** MAPE regional **18.3%** (mejora 3.3 puntos vs 21.5% municipal). En 5 de 6 regiones, el modelo regional supera al municipal.
**Lección:** Los datos agregados son más suaves y predecibles. La región es una buena capa de control y validación.

---

## El Modelo Actual: Sistema de Dos Capas

No es It7 **o** It8. Es **It7 + It8 trabajando juntos**.

```
SEMANA 1 (lunes):

El modelo REGIONAL (It8) predice:
  ├─ Noroeste: 450 unidades de SKU X
  ├─ Noreste:  920 unidades
  ├─ Centro:   580 unidades
  └─ ... (otras regiones)

El modelo MUNICIPAL (It7) predice:
  ├─ A Coruña: 45u
  ├─ Pontevedra: 38u
  ├─ Vizcaya: 125u
  ├─ Guipúzcoa: 98u
  ├─ ... (otras localidades)

VALIDACIÓN AUTOMÁTICA:
  Suma Noroeste (It7) = 45+38+...  ≈ 450 (It8)  ✓ Coherente
  Suma Noreste (It7)  = 125+98+... ≈ 920 (It8)  ✓ Coherente

Si alguna suma difiere >10%, alerta: revisar anomalía
```

### Decisiones que cada capa alimenta

**CAPA 1 — Municipal (It7):**
- "¿Cuánto envío al almacén de Bilbao?" → 125 unidades
- "¿Riesgo de rotura en Sevilla?" → Sí, histórico bajo + aumento esperado
- "¿Sobrestock en Madrid?" → Sí, predicción baja, stock físico alto

**CAPA 2 — Regional (It8):**
- "¿Cuánto compro en total para el Norte?" → 400 unidades
- "¿La región Canarias va a la baja?" → Sí, last 4 weeks trending down
- "¿Necesito rebalancear stock?" → Sí, Noreste sobre-predice, Sur bajo-predice

---

## Las Variables que Más Importan

### En Municipio (It7)
1. **Historial local (68%)**: cuánto se vende históricamente de ese SKU en esa localidad
2. **Demanda reciente (12%)**: promedio de las últimas 4 semanas
3. **Descuento promocional (5%)**: si hay campaña activa
4. **Precio (4%)**: productos baratos venden más

**En español simple:** el mejor predictor de lo que va a pasar es lo que ya ha pasado en ese mismo lugar. Si siempre vendes 20 unidades de un producto en Barcelona, probablemente venda 20 la próxima semana también.

### En Región (It8)
1. **Historial regional (60%)**: cuánto vende históricamente por SKU×región
2. **Demanda reciente regional (14%)**: promedio últimas 4 semanas a nivel región
3. **Número de municipios activos (8%)**: ¿en cuántos municipios de la región se vende?
4. **Precio (6%)**: igual que a nivel municipio

**En español simple:** a nivel región, la historia regional es más importante que a nivel municipio. Porque hay más datos que promedian el ruido local.

---

## Dónde el Modelo Funciona Mejor y Dónde Tiene Dificultades

### Verde (Funciona muy bien)
```
Noreste (Barcelona, Valencia, etc.)
├─ MAPE It7: 18.5%
├─ MAPE It8: 13.5%  ✓✓ Excelente
└─ Razón: zona de alta densidad de ventas, mucho histórico, patrones claros

Noroeste (Galicia)
├─ MAPE It7: 16.5%
├─ MAPE It8: 13.8%  ✓✓ Excelente
└─ Razón: mercado estable, pocas anomalías, productos establecidos
```

### Amarillo (Funciona aceptablemente)
```
Centro (Madrid, Toledo, etc.)
├─ MAPE It7: 25.1%
├─ MAPE It8: 21.1%  ✓ Aceptable
└─ Razón: Madrid es grande pero volátil, cambios de moda rápidos, competencia fuerte

Sur (Sevilla, Málaga, etc.)
├─ MAPE It7: 20.9%
├─ MAPE It8: 14.3%  ✓✓ Bueno
└─ Razón: más estable que Centro, mercado con patrón claro
```

### Rojo (Tiene dificultades)
```
Norte (País Vasco, Navarra, Aragón)
├─ MAPE It7: 30.5%
├─ MAPE It8: 28.1%  ⚠ Sigue siendo difícil
└─ Razón: alta demanda + alta variabilidad = combinación peligrosa.
          El modelo no sabe si la próxima semana será "normal" o "pico de 2x"

Canarias
├─ MAPE It7: 33.5%
├─ MAPE It8: 38.1%  ✗ Empeora a nivel región
└─ Razón: mercado insular con estacionalidad turística muy particular.
          El modelo entrena con patrón A pero la realidad sigue patrón B.
```

---

## Qué es un Modelo Hurdle y Por Qué Ayudaría

### El Problema Actual con B&C

Los productos de nicho (B&C) tienen demanda muy esporádica:
- Semana 1: 0 unidades (nadie compra)
- Semana 2: 5 unidades (alguien necesita)
- Semana 3: 0 unidades
- Semana 4: 12 unidades (oferta especial)
- Semana 5: 0 unidades
- ...

El modelo intenta predecir directamente estos números: "¿0, 5, 0, 12, 0, ...?" Es como intentar predecir caras y cruces de una moneda — hay demasiada incertidumbre.

### La Solución Hurdle

Un modelo hurdle es un **proceso de dos pasos**:

```
PASO 1: ¿Habrá venta esta semana o no?
        (Pregunta binaria: SÍ / NO)

PASO 2: Si la respuesta es SÍ, ¿cuánto?
        (Predicción de cantidad, pero solo sobre las semanas con venta)
```

### Ejemplo Concreto

**Producto B&C: cinturón especial de bicicleta**

**Sistema Actual (It7):**
- Historial: 20 semanas con venta, 20 semanas sin venta
- Predicción directa: "predice 2.3 unidades"
- Realidad semana siguiente: 0 unidades
- Error: −100% (predijimos venta, no la hubo)

**Sistema Hurdle:**
- PASO 1: "¿Habrá compra de cinturón esta semana?"
  - Entrada: day of year, precio actual, descuento, inventario, búsquedas web
  - Respuesta: 60% de probabilidad SÍ, 40% de probabilidad NO
- PASO 2 (si respuesta es SÍ): "¿Cuántas unidades?"
  - Solo entrenamos sobre las 20 semanas que SÍ hubo venta
  - Predicción: 4.8 unidades

**Resultado:**
- Si predicción PASO 1 es SÍ → predice 4.8 unidades
- Si predicción PASO 1 es NO → predice 0 unidades
- Error esperado: mucho menor porque el modelo ya "sabe" si va a haber demanda o no

### Beneficios Específicos del Modelo Hurdle

| Beneficio | Por Qué Importa |
|---|---|
| **Mejor en B&C** | Estos productos son "sí o no", no "cuánto". El hurdle captura eso. |
| **Reduce ceros falsos** | Hoy predice "0.5 unidades" y redondea a 0. Hurdle dice "no habrá compra" directamente. |
| **Explica la incertidumbre** | Puede decir "80% seguro de que hay venta, pero si la hay será pequeña". Más información para el comprador. |
| **Aprovecha nuevas señales** | El PASO 1 puede usar variables que el PASO 2 no necesita (ej: búsquedas web, promociones planificadas). |
| **Mejora regional** | Para regiones como Canarias (donde la venta es muy esporádica), un hurdle da mejores resultados. |

### Números Esperados con Hurdle en B&C

| Métrica | Hoy (It7) | Con Hurdle | Mejora |
|---|---|---|---|
| **MAPE B&C** | 16.75% | ~12-14% | −2.75 a 4.75 pp |
| **MAE B&C** | 0.637 | ~0.50 | −21% |
| **Aciertos en "habrá venta"** | 78% | 88% | +10 pp |

---

## Evaluación Final: ¿Estamos Listos para Producción?

### Lo que está listo
✓ **Sistema de predicción funcionando**: It7 (municipal) y It8 (regional) entrenados y validados  
✓ **Reducción de error demostrada**: −21% respecto al baseline  
✓ **Dos capas de control**: podemos validar coherencia municipal ↔ regional  
✓ **Variables identificadas**: sabemos qué importa (historial, precio, descuentos, clima)  
✓ **Modelos separados por segmento**: A y B&C optimizados cada uno

### Lo que falta antes de usar en decisiones reales
⚠ **Integración con ERP**: conectar predicciones a sistema de compra  
⚠ **Dashboard operacional**: interfaz para que el comprador vea predicciones semanales  
⚠ **Validación de impacto**: ejecutar pilot con 1-2 regiones, medir ahorro real  
⚠ **Festivos autonómicos**: datos sobre picos locales (aún no incorporados)  
⚠ **Modelo hurdle para B&C**: mejora técnica para productos esporádicos  
⚠ **Modelo específico Norte**: Canarias sigue siendo difícil, merece atención especial

### ¿En cuánto tiempo?
- **2-4 semanas**: Dashboard + integración ERP
- **4-8 semanas**: Pilot + validación de impacto
- **8-12 semanas**: Hurdle + modelo regional específico
- **12-16 semanas**: Producción completa + monitoreo

---

## Conclusión: El Impacto Real en el Negocio

### En Dinero
Si cada error de predicción de 100 unidades cuesta €500 (sobrestock o venta perdida):
- **Baseline (It1)**: Error medio 0.793 u/semana × 250K predicciones = 198K unidades error/año = **€99M riesgo**
- **Hoy (It7)**: Error medio 0.628 u/semana × 250K predicciones = 157K unidades error/año = **€78.5M riesgo**
- **Ahorro por mejor predicción**: **€20.5M/año** (21% de reducción)

Obviamente, el error no se distribuye uniformemente. Pero incluso si solo capturaos el 10% de ese potencial, estamos hablando de **€2M/año en ahorro**.

### En Operaciones
- Menos roturas de stock → más ventas completadas
- Menos sobrestock → menos capital muerto
- Más transparencia → el comprador sabe dónde hay riesgo

### En Aprendizaje
- Hemos descubierto que descuentos = ×4.2 en demanda
- Sabemos que el Norte es 2× más volátil que Noroeste
- El modelo enseña que el historial local lo explica casi todo

---

## Próximos Pasos Inmediatos

| Semana | Acción | Responsable |
|---|---|---|
| 1-2 | Crear dashboard prototipo con It7 + It8 | Equipo técnica |
| 2-3 | Presentar pilot a equipo de compras | Producto |
| 3-4 | Ejecutar pilot en Noroeste (zona de éxito) | Compras + técnica |
| 4-6 | Medir impacto: error real vs predicción | Producto |
| 6-8 | Escalar a Noreste si pilot es positivo | Compras + técnica |

**Recomendación:** No esperar perfección. La predicción actual (21.5% MAPE) es funcional. Lanzar pilot ahora, mejorar iterativamente en producción.

---

## Apéndice: Glosario para Ejecutivos

| Término | Significa |
|---|---|
| **MAE** | Error Promedio. Si MAE=0.628, nos equivocamos en promedio 0.628 unidades. |
| **MAPE** | Error en Porcentaje. Si MAPE=21%, el error es el 21% del valor real promedio. |
| **R²** | Qué % del comportamiento explica el modelo. R²=0.3 = explica el 30%. |
| **It7, It8** | Iteración 7, Iteración 8. Versiones del modelo, cada una con mejoras. |
| **SKU** | Stock Keeping Unit. Código del producto (ej: "cinturón rojo talla M"). |
| **Overfitting** | Cuando el modelo memoriza en lugar de aprender. Funciona en datos conocidos pero falla en nuevos. |
| **Target encoding** | Resumir el historial de cada grupo en una sola variable. |
| **Hurdle** | Modelo de dos pasos: primero "sí/no", luego "cuánto". |

# EVALUACIÓN EXHAUSTIVA — PROYECTO CRUZBER MDA
### Análisis de Head of Data • Marzo 2026

---

## 1. ¿ES EL MEJOR MODELO QUE UTILIZARÍA?

**Respuesta corta: CatBoost es la elección correcta, pero el problema está mal encuadrado en su núcleo para los SKUs tipo B/C.**

### Lo que está bien elegido
- **CatBoost** es el algoritmo ideal para este tipo de datos tabulares con categorías de alta cardinalidad (municipio, SKU). Su manejo nativo de categóricas y robustez a outliers es perfecto para ventas de retail.
- La **arquitectura de dos modelos** (A vs B&C) es un acierto sólido: tipo A tiene comportamiento continuo, tipo B&C es esencialmente esporádico.
- El **enfoque iterativo y metodológico** es exactamente como se debe desarrollar un modelo de predicción de demanda en un entorno real.

### Lo que cambiaría o añadiría

| Problema | Impacto | Alternativa |
|---|---|---|
| Un solo modelo de regresión para todo | Los SKUs tipo B/C con 80% ceros no son un problema de regresión pura | **Modelo Hurdle (Tweedie o Zero-Inflated)** para B/C |
| Overfitting estructural del 34-48% | No es ajustable con regularización, es información que falta | Aceptado como límite del dataset, bien diagnosticado |
| R² de 0.29-0.45 | El modelo explica solo 29-45% de la varianza | Normal para demanda retail — no es un fallo del modelo |
| Log1p pero sin Tweedie loss | La transformación log ayuda, pero una distribución Tweedie nativa captaría mejor la cola larga | Probar `loss_function='Tweedie'` en CatBoost |
| Sin modelo de Canarias dedicado | MAPE 38-41% es inaceptable para producción | Modelo separado con features específicas (turismo, estacionalidad isleña) |

### ¿Hay algo mejor que CatBoost con estos datos?
Con este dataset no cambiaría de algoritmo, pero sí cambiaría la **formulación del problema** para SKUs B/C:

```
Modelo actual:   predict(unidades_t) ← regresión directa
Mejor opción:    P(venta > 0) × E[unidades | venta > 0]
                    ↑                    ↑
                clasificación         regresión solo sobre positivos
                (¿habrá venta?)      (¿cuánto si hay?)
```

---

## 2. EVALUACIÓN PASO A PASO POR ITERACIÓN

| Iteración | Descripción | Nota Técnica | Comentario |
|---|---|---|---|
| **Notebooks 01-02** | Data prep y feature engineering base | 8.5/10 | Muy limpio. Bien justificadas las exclusiones. Falta análisis de zero-inflation (% ceros por SKU) |
| **It1 — Baseline** | Comparativa de 4 algoritmos | 9/10 | Excelente punto de partida. Split temporal correcto. CatBoost bien elegido |
| **It2 — Rolling Mean** | Media móvil 4 semanas | 7.5/10 | Feature correcta y shift anti-leakage bien aplicado. Podría explorar más ventanas (2, 8, 12 sem) |
| **It3 — Año anterior** | Seasonal lag interanual | 6.5/10 | Correcta en concepto, pero con solo 1 año de historia la cobertura es baja. El aprendizaje es el valor |
| **It4 — Log1p** | Transformación de la target | **10/10** | **La mejor decisión del proyecto.** −15.6% de MAE de un solo cambio conceptual. Demuestra madurez analítica real |
| **It5 — Optuna + Target Encoding** | Hiperparámetros + historia por ubicación | 9/10 | Target encoding con validación temporal sin leakage es complejo de hacer bien — se hizo correctamente |
| **It6 — Descuentos + Región** | Señal promocional + geografía | 8/10 | El ×4.7 de efecto descuento es el hallazgo más valioso del proyecto. Mejora pequeña en MAE porque cobertura solo 1.5% |
| **It7 — Modelos dedicados A vs B/C** | Especialización por tipo de producto | 9.5/10 | Decisión arquitectónica excelente. Parámetros radicalmente distintos (lr 0.187 vs 0.051) validan que son problemas diferentes |
| **It8 — Agregación regional** | Predicción a nivel región | 8.5/10 | Brillante la idea de arquitectura de dos capas como sistema de validación cruzada. Canarias bien identificado como excepción |

---

## 3. NOTA GLOBAL: **7.8 / 10**

### Desglose por dimensión

| Dimensión | Nota | Justificación |
|---|---|---|
| **Metodología y rigor** | 9/10 | Split temporal correcto, CV time series, sin leakage |
| **Selección de algoritmos** | 8/10 | CatBoost es la elección óptima. Falta Tweedie/Hurdle para B/C |
| **Feature engineering** | 7.5/10 | Buenas features, pero faltan algunas claramente disponibles con los datos actuales |
| **Interpretabilidad** | 8.5/10 | Feature importance bien analizada, decisiones justificadas |
| **Iteratividad y aprendizaje** | 9/10 | Cada iteración aprende y construye sobre la anterior. Estructura ejemplar |
| **Diagnóstico de problemas** | 8.5/10 | Overfitting estructural bien identificado, Canarias detectada como outlier |
| **Reproducibilidad** | 7/10 | Artifacts exportados, pero falta pipeline de inferencia documentado |
| **Impacto de negocio** | 6/10 | Estimaciones de €20M son optimistas sin validación con el negocio real |

---

## 4. PASOS PARA MEJORAR EL MODELO (SOLO CON LOS DATASETS EXISTENTES)

### A. Mejoras Inmediatas — alta ROI, poco esfuerzo

**A1. Hurdle Model para SKUs tipo B/C**

```python
# Paso 1: Clasificador — ¿habrá venta esta semana?
modelo_clas = CatBoostClassifier(...)  # target: (unidades > 0)

# Paso 2: Regresor — ¿cuánto si hay venta? (solo sobre filas con venta positiva)
modelo_reg = CatBoostRegressor(...)   # entrenar solo sobre rows donde unidades > 0

# Predicción final:
pred = P_venta * pred_cantidad
```

El dataset ya tiene toda la información necesaria. Esperado: **−3 a −5 puntos de MAPE en B/C**.

**A2. Tweedie Loss Function en CatBoost**

```python
# Cambio de una línea, cero datos nuevos
CatBoostRegressor(loss_function='Tweedie:variance_power=1.5', ...)
```

Nativo para distribuciones con muchos ceros y cola larga. Muy indicado para B/C.

**A3. Features temporales que faltan — derivables gratis del dataset**

Con `anio` y `semana_anio` que ya existen:

```python
df['mes']              = semana_to_mes(df['semana_anio'])
df['trimestre']        = ...
df['semana_del_mes']   = ...         # ¿es la 1ª, 2ª, 3ª, 4ª semana del mes?
df['es_fin_mes']       = ...         # últimos 7 días del mes (spikes de cierre)
df['sin_semana']       = np.sin(2*np.pi*df.semana_anio / 52)  # estacionalidad cíclica
df['cos_semana']       = np.cos(2*np.pi*df.semana_anio / 52)
```

**A4. Más lags y más ventanas de rolling**

El dataset tiene lag_1w y lag_4w. Añadir:

```python
lags            = [2, 3, 6, 8, 12, 26, 52]
rolling_windows = [2, 6, 12, 26]
rolling_std     = [4, 8, 12]          # volatilidad: ¿está el SKU en racha o errático?
```

**A5. Features de tendencia local**

```python
# Tendencia de las últimas 4 semanas vs las 4 anteriores
df['tendencia_4v4'] = rolling_4 / (rolling_8 - rolling_4 + 1)

# Ratio vs mismo periodo año anterior (lag_anual ya existe)
df['ratio_yoy'] = df['unidades'] / (df['unidades_ano_anterior'] + 0.1)
```

### B. Mejoras Estructurales — esfuerzo medio, alto impacto

**B1. Modelo específico para Canarias**

El dataset ya tiene `Provincia`. Canarias tiene dinámicas radicalmente distintas (turismo, estacionalidad invertida en algunos periodos). Un modelo separado podría bajar su MAPE del 38% al 22-25%.

**B2. Detección automática de semanas atípicas**

Las semanas 9, 19 y 36 tienen MAE +50% superior a la media. Con el histórico disponible:

```python
df['es_semana_problematica'] = df['semana_anio'].isin([9, 19, 36]).astype(int)
df['dist_semana_problematica'] = df['semana_anio'].apply(
    lambda w: min(abs(w-9), abs(w-19), abs(w-36))
)
```

**B3. Interacciones cruzadas con el dataset de ciclismo**

```python
# ¿Hay prueba ciclista la semana siguiente? (efecto anticipatorio de compra)
df['prueba_prox_semana'] = lag_futuro_1(num_pruebas_ciclistas)

# Intensidad del evento
df['intensidad_evento'] = num_pruebas * duracion_total
```

**B4. Price elasticity features desde el propio dataset**

Con `importe_neto` y `unidades` ya disponibles:

```python
df['precio_unitario']       = df['importe_neto'] / df['unidades'].clip(1)
df['precio_vs_media_sku']   = precio_unitario / precio_medio_sku   # ¿está en oferta?
df['precio_vs_media_region'] = ...
```

### C. Mejoras de Validación y Producción

**C1. Walk-forward validation más rigurosa**

Con el dataset actual conviene hacer walk-forward mensual sobre 2024 completo: entrenar hasta marzo → predecir abril; entrenar hasta abril → predecir mayo... Esto da un benchmark de producción real mucho más fiable.

**C2. Predicción a horizonte múltiple**

El modelo actual predice semana t+1. Negocio necesita 4-8 semanas de horizonte para compras:

```python
# Direct multi-step: un modelo por horizonte (más preciso, más costoso)
modelo_t1, modelo_t2, modelo_t4, modelo_t8 = ...

# O recursive: usar predicción de t+1 como feature para t+2 (más simple, acumula error)
```

---

## 5. ¿SERÍAN INTERESANTES MODELOS PREVIOS DE CLUSTERIZACIÓN?

**Sí, rotundamente. Este es probablemente el siguiente gran salto disponible con los datos actuales.**

### Clusterización de SKUs ★★★ — Alta prioridad

Ahora el modelo trata cada SKU como independiente. Un clustering de SKUs en 8-15 grupos según su patrón de demanda sería una feature de contexto extremadamente rica.

```python
# Features para clustering de SKUs — todo derivable de df_final_modelado.csv:
sku_features = {
    'vol_medio_semanal':        ...,
    'cv_demanda':               std/mean,       # ¿errático o estable?
    'pct_semanas_con_venta':    ...,             # ¿producto de nicho o masivo?
    'concentracion_geografica': ...,             # ¿vende en todo España o 2 provincias?
    'semana_pico':              ...,             # ¿cuándo se vende más?
    'tendencia_yoy':            ...,             # ¿creciente, decreciente, estable?
    'sensibilidad_descuento':   ...,             # ratio de venta con/sin promo
}
```

El cluster asignado a cada SKU se añade como feature al modelo principal. El modelo aprende que "SKUs del cluster 3 (equipamiento pesado) tienen pico semanas 15-25 y responden fuerte a descuentos".

**Ganancia estimada: −2 a −4 puntos de MAPE para tipo B/C.**

### Clusterización de Municipios ★★☆ — Media prioridad

Con 1000+ municipios, muchos tienen historia muy escasa. Clusters de municipios con comportamiento similar aportarían señal donde no hay historia suficiente (cold start problem).

```python
muni_features = {
    'demanda_total':            ...,
    'num_skus_activos':         ...,
    'mix_abc':                  proporcion_A / total,
    'efecto_estacional':        cv_mensual,
    'respuesta_clima':          corr(unidades, temp_media),
    'densidad_ciclista':        eventos_año_provincia,
}
```

### Clusterización Temporal (Regímenes de Demanda) ★★☆ — Media prioridad

La más infrautilizada y potencialmente muy potente:

```python
# Identificar regímenes temporales automáticamente
# No hardcodear "semanas 9, 19, 36 son problemáticas"
# sino aprender: "semanas 48-52 + 1-3 = régimen navideño"
#               "semanas 14-25 = régimen ciclista"
# Input: clustering de semanas × perfil de ventas (media, varianza, % descuentos)
```

### Segmentación de Clientes ★★★ — Alta prioridad a largo plazo

No disponible en los datasets actuales, pero sería la más potente. Si Cruzber tiene CRM o datos de canal B2B vs B2C con identificador de cliente, es el siguiente gran dataset a incorporar.

### Tabla Resumen de Clustering

| Tipo | Complejidad | Ganancia Esperada | Datos necesarios |
|---|---|---|---|
| **Clustering SKUs** | Media | Alta — −2 a −4 pp MAPE | Solo `df_final_modelado.csv` |
| **Clustering Municipios** | Media | Media — ayuda en cold start | Solo `df_final_modelado.csv` |
| **Clustering Temporal** | Baja | Media — mejor que hardcodear semanas | Solo `df_final_modelado.csv` |
| **Segmentación Clientes** | Alta | Muy alta (largo plazo) | No disponible actualmente |

---

## RESUMEN EJECUTIVO

El trabajo tiene una **calidad técnica genuinamente buena**, especialmente en la solidez metodológica: no hay leakage, el split es temporal, las iteraciones aprenden. El salto más brillante del proyecto fue **It4 con log1p** — demuestra que el equipo entiende la naturaleza del problema, no solo aplica algoritmos.

**Las tres áreas de mejora prioritarias, con los datos disponibles:**

1. **Hurdle model para B/C** — reformulación del problema, no más features
2. **Clustering de SKUs como feature** — contexto que el modelo aún no tiene
3. **Más lags y rolling features** — señales de momentum derivables gratis del dataset

El modelo está **listo para un piloto en producción** en regiones con buen desempeño (Noroeste, Noreste), pero **no debería desplegarse para Canarias y Norte** sin un modelo dedicado o datos adicionales.
